# Room 3.9 — Data Preprocessing

This notebook prepares the dataset used for the analysis of Room 3.9.

The preprocessing pipeline processes the available sensor measurements collected inside the room and transforms the raw telemetry into a consistent 10-minute temporal representation.

The main preprocessing steps include:

- loading and restructuring the raw sensor telemetry,
- identification and removal of invalid measurements,
- temporal aggregation into 10-minute intervals,
- handling of missing observations,
- creation of derived variables used in the subsequent analysis.

The final processed datasets produced in this notebook are used directly in the exploratory analysis and subsequent modelling stages of the thesis.

## 1. Indoor Air Quality (IAQ) Preprocessing

The IAQ sensor installed in Room 3.9 records five environmental and occupancy-related variables:

- CO₂ concentration,
- temperature,
- relative humidity,
- light level,
- PIR-based motion detection.

The raw telemetry is first converted from JSON format into a long-format dataset. Invalid CO₂ measurements are then identified and removed before the measurements are aggregated into common 10-minute intervals.

The resulting dataset provides the environmental variables used in the subsequent analysis of Room 3.9.

In [1]:
import json

import numpy as np
import pandas as pd

In [2]:
# Load the raw IAQ telemetry for Room 3.9.

with open(
    "iaq_3_9_raw.json",
    "r",
    encoding="utf-8"
) as file:
    data = json.load(file)


# Display the available telemetry variables.

print("Telemetry keys:")
print(data.keys())


# Convert the JSON telemetry into long format.

dfs = []

for key, values in data.items():

    temp_df = pd.DataFrame(values)

    temp_df["key"] = key

    dfs.append(temp_df)


df_long = pd.concat(
    dfs,
    ignore_index=True
)


# Convert timestamps from Unix milliseconds to datetime.

df_long["ts"] = pd.to_datetime(
    df_long["ts"],
    unit="ms"
)


# Convert sensor measurements to numeric values.

df_long["value"] = pd.to_numeric(
    df_long["value"],
    errors="coerce"
)


print("\nLong-format dataset shape:")
print(df_long.shape)

display(df_long.head())

Telemetry keys:
dict_keys(['co2', 'temperature', 'humidity', 'light_level', 'pir'])

Long-format dataset shape:
(195249, 3)


,ts,value,key
0,2026-05-28 17:06:27.395,65535.0,co2
1,2026-05-28 17:06:11.942,65535.0,co2
2,2026-05-28 16:54:58.965,500.0,co2
3,2026-05-28 16:44:59.189,523.0,co2
4,2026-05-28 16:34:59.324,608.0,co2


### Identification and Removal of Invalid Measurements

Before temporal aggregation, the raw telemetry is inspected for invalid sensor readings.

For the CO₂ measurements, the value `65535` represents an invalid sensor state and is therefore excluded from the subsequent analysis. Removing these measurements prevents sensor-error values from affecting the calculated 10-minute environmental observations.

In [3]:
# Identify invalid CO₂ measurements.
# The value 65535 corresponds to an invalid sensor reading.

invalid_co2_mask = (
    (df_long["key"] == "co2")
    &
    (df_long["value"] == 65535)
)

invalid_co2_count = invalid_co2_mask.sum()

print(
    "Invalid CO2 readings:",
    invalid_co2_count
)

Invalid CO2 readings: 519


In [4]:
# Remove invalid CO₂ measurements while preserving
# all valid observations from the remaining variables.

df_long_clean = (
    df_long
    .loc[~invalid_co2_mask]
    .copy()
    .reset_index(drop=True)
)

print(
    "Shape after removing invalid CO2 readings:",
    df_long_clean.shape
)

print("\nMissing values:")
print(
    df_long_clean.isna().sum()
)

Shape after removing invalid CO2 readings: (194730, 3)

Missing values:
ts       0
value    0
key      0
dtype: int64


### Temporal Aggregation to 10-Minute Intervals

The cleaned telemetry is aligned to common 10-minute intervals in order to create a consistent time-series representation of the indoor environmental conditions.

Each raw observation is assigned to the corresponding 10-minute interval. Measurements of the same variable within the same interval are then aggregated using their mean value.

The resulting wide-format dataset contains one row per 10-minute timestamp and one column for each recorded variable.

In [5]:
# Assign each cleaned observation to a 10-minute interval.

df_long_clean["Timestamp"] = (
    df_long_clean["ts"]
    .dt.floor("10min")
)

In [6]:
# Transform the long-format telemetry into a wide-format
# dataset containing one row per 10-minute interval.

df_wide = (
    df_long_clean
    .pivot_table(
        index="Timestamp",
        columns="key",
        values="value",
        aggfunc="mean"
    )
    .reset_index()
)

df_wide.columns.name = None


print(
    "10-minute dataset shape:",
    df_wide.shape
)

display(
    df_wide.head()
)

10-minute dataset shape: (38651, 6)


,Timestamp,co2,humidity,light_level,pir,temperature
0,2025-08-31 21:00:00,419.0,41.5,0.0,0.0,29.9
1,2025-08-31 21:10:00,419.0,41.5,0.0,0.0,29.9
2,2025-08-31 21:20:00,420.0,41.5,0.0,0.0,29.9
3,2025-08-31 21:30:00,420.0,41.5,0.0,0.0,29.9
4,2025-08-31 21:40:00,418.0,41.5,0.0,0.0,29.9


### Validation of the 10-Minute Dataset

Before producing the final IAQ dataset, the temporally aggregated data are inspected for completeness and consistency.

The validation includes the temporal coverage of the dataset, missing values in the main environmental variables, duplicate timestamps and descriptive statistics of the CO₂ measurements.

In [8]:
# Validate the 10-minute IAQ dataset.

print(
    "Shape:",
    df_wide.shape
)

print(
    "\nTime range:",
    df_wide["Timestamp"].min(),
    "→",
    df_wide["Timestamp"].max()
)

print("\nMissing values:")
print(
    df_wide[
        [
            "co2",
            "temperature",
            "humidity"
        ]
    ]
    .isna()
    .sum()
)

print("\nDuplicate timestamps:")
print(
    df_wide["Timestamp"]
    .duplicated()
    .sum()
)

print("\nCO2 statistics:")
print(
    df_wide["co2"].describe()
)

Shape: (38651, 6)

Time range: 2025-08-31 21:00:00 → 2026-05-28 17:00:00

Missing values:
co2            95
temperature     0
humidity        0
dtype: int64

Duplicate timestamps:
0

CO2 statistics:
count    38556.000000
mean       490.866355
std        156.202260
min        378.000000
25%        419.000000
50%        443.000000
75%        495.000000
max       3101.000000
Name: co2, dtype: float64


### Final Cleaning and CO₂-Level Classification

The remaining 10-minute intervals without a valid CO₂ measurement are excluded from the final dataset. This ensures that every retained observation contains a valid CO₂ concentration together with the corresponding environmental measurements.

For descriptive analysis, CO₂ concentration is additionally classified into three levels:

- **Low:** up to 500 ppm,
- **Medium:** above 500 and up to 700 ppm,
- **High:** above 700 ppm.

This categorical variable is retained alongside the original continuous CO₂ measurements.

In [9]:
# Remove the remaining intervals without a valid CO₂ measurement.

df_iaq_3_9_clean = (
    df_wide
    .dropna(
        subset=["co2"]
    )
    .copy()
    .reset_index(drop=True)
)


# Create the categorical CO₂-level variable.

df_iaq_3_9_clean["co2_level"] = pd.cut(
    df_iaq_3_9_clean["co2"],
    bins=[
        0,
        500,
        700,
        float("inf")
    ],
    labels=[
        "Low",
        "Medium",
        "High"
    ],
    include_lowest=True
)


print("CO2 level distribution:")
print(
    df_iaq_3_9_clean["co2_level"]
    .value_counts()
)

CO2 level distribution:
co2_level
Low       29550
Medium     6439
High       2567
Name: count, dtype: int64


### Final Dataset Validation and Export

The final IAQ dataset is validated before export to ensure that it contains no missing values or duplicate timestamps and that the expected temporal coverage has been preserved.

The cleaned dataset is then exported as a CSV file for use in the subsequent analysis of Room 3.9.

In [10]:
# Final validation of the cleaned Room 3.9 IAQ dataset.

print(
    "Final shape:",
    df_iaq_3_9_clean.shape
)

print("\nMissing values:")
print(
    df_iaq_3_9_clean
    .isna()
    .sum()
)

print("\nDuplicate timestamps:")
print(
    df_iaq_3_9_clean["Timestamp"]
    .duplicated()
    .sum()
)

print("\nTime range:")
print(
    df_iaq_3_9_clean["Timestamp"].min(),
    "→",
    df_iaq_3_9_clean["Timestamp"].max()
)

display(
    df_iaq_3_9_clean.head()
)

Final shape: (38556, 7)

Missing values:
Timestamp      0
co2            0
humidity       0
light_level    0
pir            0
temperature    0
co2_level      0
dtype: int64

Duplicate timestamps:
0

Time range:
2025-08-31 21:00:00 → 2026-05-28 16:50:00


,Timestamp,co2,humidity,light_level,pir,temperature,co2_level
0,2025-08-31 21:00:00,419.0,41.5,0.0,0.0,29.9,Low
1,2025-08-31 21:10:00,419.0,41.5,0.0,0.0,29.9,Low
2,2025-08-31 21:20:00,420.0,41.5,0.0,0.0,29.9,Low
3,2025-08-31 21:30:00,420.0,41.5,0.0,0.0,29.9,Low
4,2025-08-31 21:40:00,418.0,41.5,0.0,0.0,29.9,Low


In [11]:
# Export the cleaned IAQ dataset.

df_iaq_3_9_clean.to_csv(
    "iaq_3_9_clean.csv",
    index=False
)

print(
    "Saved: iaq_3_9_clean.csv"
)

print(
    "Final dataset shape:",
    df_iaq_3_9_clean.shape
)

Saved: iaq_3_9_clean.csv
Final dataset shape: (38556, 7)


## 2. People Counter Preprocessing

Room 3.9 contains a People Counter sensor that records the flow of people entering and leaving the room.

The available telemetry includes cumulative and period-based counters for incoming and outgoing movements. The raw measurements are converted into a common 10-minute representation so that they can later be combined with the environmental measurements of the room.

The preprocessing procedure includes:

- loading the raw People Counter JSON telemetry,
- converting timestamps and counter values,
- combining the different telemetry variables into a long-format dataset,
- aligning the measurements to 10-minute intervals,
- transforming the dataset into wide format,
- validating the resulting timestamps and missing values.

The cleaned dataset is exported as:

`people_counter_3_9_clean.csv`

In [12]:
# PEOPLE COUNTER PREPROCESSING — ROOM 3.9


with open(
    "people_counter_3_9_raw.json",
    "r",
    encoding="utf-8"
) as file:
    data = json.load(file)


print("Telemetry keys:")
print(data.keys())


# Inspect the first observations of each telemetry variable.

for key in data.keys():

    print(f"\n{key}")

    display(
        pd.DataFrame(
            data[key]
        ).head()
    )

Telemetry keys:
dict_keys(['line_1_period_in', 'line_1_period_out', 'line_1_total_in', 'line_1_total_out'])

line_1_period_in


,ts,value
0,1783285031289,0
1,1783285026290,0
2,1783284429577,0
3,1783284426316,0
4,1783283830320,0



line_1_period_out


,ts,value
0,1783285031289,0
1,1783285026290,0
2,1783284429577,0
3,1783284426316,0
4,1783283830320,0



line_1_total_in


,ts,value
0,1783285031289,1
1,1783285026290,1
2,1783284429577,1
3,1783284426316,1
4,1783283830320,1



line_1_total_out


,ts,value
0,1783285031289,1
1,1783285026290,1
2,1783284429577,1
3,1783284426316,1
4,1783283830320,1


In [13]:
# 2.1 Convert raw telemetry to long format


dfs = []

for key, values in data.items():

    temp_df = pd.DataFrame(values)

    temp_df["key"] = key

    dfs.append(temp_df)


df_pc = pd.concat(
    dfs,
    ignore_index=True
)


# Convert timestamps from Unix milliseconds to datetime.

df_pc["ts"] = pd.to_datetime(
    df_pc["ts"],
    unit="ms"
)


# Convert counter measurements to numeric format.

df_pc["value"] = pd.to_numeric(
    df_pc["value"]
)


print(
    "Long-format People Counter shape:",
    df_pc.shape
)

display(
    df_pc.head()
)

Long-format People Counter shape: (248088, 3)


,ts,value,key
0,2026-07-05 20:57:11.289,0,line_1_period_in
1,2026-07-05 20:57:06.290,0,line_1_period_in
2,2026-07-05 20:47:09.577,0,line_1_period_in
3,2026-07-05 20:47:06.316,0,line_1_period_in
4,2026-07-05 20:37:10.320,0,line_1_period_in


In [14]:
# 2.2 Inspect available counter variables


print(
    df_pc["key"]
    .value_counts()
)

key
line_1_total_out     62023
line_1_total_in      62023
line_1_period_out    62021
line_1_period_in     62021
Name: count, dtype: int64


### Temporal Aggregation to 10-Minute Intervals

The People Counter measurements are aligned to the same 10-minute temporal resolution used for the remaining Room 3.9 sensor datasets.

For each counter variable and 10-minute interval, the latest available value is retained. This is appropriate for cumulative and state-like counter measurements, where the most recent observation represents the current counter value at that point in time.

In [15]:
# 2.3 Temporal alignment to 10-minute intervals

df_pc["Timestamp_10min"] = (
    df_pc["ts"]
    .dt.floor("10min")
)


df_pc_wide = (
    df_pc
    .pivot_table(
        index="Timestamp_10min",
        columns="key",
        values="value",
        aggfunc="last"
    )
    .reset_index()
)

df_pc_wide.columns.name = None


print(
    "10-minute People Counter shape:",
    df_pc_wide.shape
)

display(
    df_pc_wide.head()
)

10-minute People Counter shape: (44051, 5)


,Timestamp_10min,line_1_period_in,line_1_period_out,line_1_total_in,line_1_total_out
0,2025-08-31 21:00:00,0,0,0,0
1,2025-08-31 21:10:00,0,0,0,0
2,2025-08-31 21:20:00,0,0,0,0
3,2025-08-31 21:30:00,0,0,0,0
4,2025-08-31 21:40:00,0,0,0,0


In [16]:
# 2.4 Inspect People Counter measurements

print("Total incoming counter:")
print(
    df_pc_wide[
        "line_1_total_in"
    ].describe()
)

print("\nTotal outgoing counter:")
print(
    df_pc_wide[
        "line_1_total_out"
    ].describe()
)

print(
    "\nMaximum total outgoing count:",
    df_pc_wide[
        "line_1_total_out"
    ].max()
)

Total incoming counter:
count    44051.000000
mean        13.308620
std         28.855709
min          0.000000
25%          0.000000
50%          0.000000
75%          9.000000
max        200.000000
Name: line_1_total_in, dtype: float64

Total outgoing counter:
count    44051.000000
mean        12.462804
std         27.726729
min          0.000000
25%          0.000000
50%          0.000000
75%          7.000000
max        191.000000
Name: line_1_total_out, dtype: float64

Maximum total outgoing count: 191


In [17]:
# Inspect intervals containing incoming events.

incoming_events = (
    df_pc_wide[
        df_pc_wide[
            "line_1_period_in"
        ] > 0
    ][
        [
            "Timestamp_10min",
            "line_1_period_in"
        ]
    ]
)

display(
    incoming_events.head()
)


# Inspect intervals containing outgoing events.

outgoing_events = (
    df_pc_wide[
        df_pc_wide[
            "line_1_period_out"
        ] > 0
    ][
        [
            "Timestamp_10min",
            "line_1_period_out"
        ]
    ]
)

display(
    outgoing_events.head()
)

,Timestamp_10min,line_1_period_in
75,2025-09-01 09:30:00,1
503,2025-09-04 09:20:00,2
505,2025-09-04 09:40:00,2
530,2025-09-04 13:50:00,2
618,2025-09-05 04:30:00,1


,Timestamp_10min,line_1_period_out
75,2025-09-01 09:30:00,1
504,2025-09-04 09:30:00,2
505,2025-09-04 09:40:00,2
530,2025-09-04 13:50:00,2
618,2025-09-05 04:30:00,1


### Final Validation and Export

The temporally aligned People Counter dataset is checked for temporal coverage, missing values and duplicate timestamps before being exported for subsequent integration with the remaining Room 3.9 sensor data.

In [18]:
# 2.5 Final People Counter validation

print(
    "Shape:",
    df_pc_wide.shape
)

print("\nColumns:")
print(
    df_pc_wide.columns.tolist()
)

print("\nTime range:")
print(
    df_pc_wide[
        "Timestamp_10min"
    ].min(),
    "→",
    df_pc_wide[
        "Timestamp_10min"
    ].max()
)

print("\nMissing values:")
print(
    df_pc_wide.isna().sum()
)

print("\nDuplicate timestamps:")
print(
    df_pc_wide[
        "Timestamp_10min"
    ]
    .duplicated()
    .sum()
)

Shape: (44051, 5)

Columns:
['Timestamp_10min', 'line_1_period_in', 'line_1_period_out', 'line_1_total_in', 'line_1_total_out']

Time range:
2025-08-31 21:00:00 → 2026-07-05 20:50:00

Missing values:
Timestamp_10min      0
line_1_period_in     0
line_1_period_out    0
line_1_total_in      0
line_1_total_out     0
dtype: int64

Duplicate timestamps:
0


In [19]:
# 2.6 Rename timestamp and export


df_pc_wide = (
    df_pc_wide
    .rename(
        columns={
            "Timestamp_10min": "ts"
        }
    )
)


output_file = (
    "people_counter_3_9_clean.csv"
)

df_pc_wide.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)


print(
    f"Saved: {output_file}"
)

print(
    "Final shape:",
    df_pc_wide.shape
)

display(
    df_pc_wide.head()
)

Saved: people_counter_3_9_clean.csv
Final shape: (44051, 5)


,ts,line_1_period_in,line_1_period_out,line_1_total_in,line_1_total_out
0,2025-08-31 21:00:00,0,0,0,0
1,2025-08-31 21:10:00,0,0,0,0
2,2025-08-31 21:20:00,0,0,0,0
3,2025-08-31 21:30:00,0,0,0,0
4,2025-08-31 21:40:00,0,0,0,0


## 3. Magnetic Contact Sensor Preprocessing

Room 3.9 contains three magnetic contact sensors used to monitor the state of the room windows.

Each sensor records the binary state of one monitored opening. Since magnetic contact sensors report changes in state rather than continuous measurements, their observations are resampled to 10-minute intervals and the latest known state is propagated forward until a new measurement is recorded.

The preprocessing procedure includes:

- loading the three raw Magnetic Contact JSON files,
- converting timestamps and sensor values,
- resampling each sensor independently to 10-minute intervals,
- combining the three sensors into a common dataset,
- restricting the dataset to the period where all three sensors have become available,
- calculating the total number of open windows.

The cleaned dataset is exported as:

`magnetic_contacts_3_9_clean.csv`

In [21]:
# IMPORTS


import json
import os

import numpy as np
import pandas as pd

In [22]:
# MAGNETIC CONTACT SENSOR PREPROCESSING — ROOM 3.9

files_mc = {
    "mc_1": "/content/F3_3.9-MC-1_raw.json",
    "mc_2": "/content/F3_3.9-MC-2_raw.json",
    "mc_3": "/content/F3_3.9-MC-3_raw.json"
}


# Verify that all three raw JSON files are available.

for name, path in files_mc.items():

    print(
        name,
        "->",
        os.path.exists(path),
        path
    )

mc_1 -> True /content/F3_3.9-MC-1_raw.json
mc_2 -> True /content/F3_3.9-MC-2_raw.json
mc_3 -> True /content/F3_3.9-MC-3_raw.json


In [23]:
# 3.1 Load individual Magnetic Contact sensors


def load_magnetic_contact(
    json_path,
    column_name
):
    """
    Load one Magnetic Contact JSON file and return
    a timestamped DataFrame containing its binary state.
    """

    with open(
        json_path,
        "r",
        encoding="utf-8"
    ) as file:

        data = json.load(file)

    records = data.get(
        "magnet_status",
        []
    )

    df = pd.DataFrame(records)

    # Return an empty structured DataFrame if no measurements
    # are available.
    if df.empty:

        return pd.DataFrame(
            columns=[
                "ts",
                column_name
            ]
        )

    # Convert Unix timestamps from milliseconds to datetime.
    df["ts"] = pd.to_datetime(
        df["ts"],
        unit="ms"
    )

    # Convert sensor state to numeric format.
    df["value"] = pd.to_numeric(
        df["value"],
        errors="coerce"
    )

    df = df.rename(
        columns={
            "value": column_name
        }
    )

    df = (
        df[
            [
                "ts",
                column_name
            ]
        ]
        .sort_values("ts")
        .reset_index(drop=True)
    )

    return df

In [24]:
# Load the three Magnetic Contact sensors.

mc_1 = load_magnetic_contact(
    files_mc["mc_1"],
    "mc_1"
)

mc_2 = load_magnetic_contact(
    files_mc["mc_2"],
    "mc_2"
)

mc_3 = load_magnetic_contact(
    files_mc["mc_3"],
    "mc_3"
)


# Inspect the temporal coverage of each sensor.

for name, df in {
    "mc_1": mc_1,
    "mc_2": mc_2,
    "mc_3": mc_3
}.items():

    print(
        f"{name}: {len(df)} observations | "
        f"{df['ts'].min()} → {df['ts'].max()}"
    )

mc_1: 2444 observations | 2025-08-31 23:24:48.866000 → 2026-07-05 17:46:42.415000
mc_2: 629 observations | 2025-08-31 23:35:21.024000 → 2026-07-05 17:54:59.370000
mc_3: 594 observations | 2025-08-31 23:43:26.656000 → 2026-07-05 18:00:58.179000


### Temporal Alignment to 10-Minute Intervals

Each Magnetic Contact sensor is independently resampled to 10-minute intervals. The most recent state within each interval is retained and then propagated forward, reflecting the assumption that a window remains in its latest known state until a new change is recorded.

In [25]:
# 3.2 Resample each sensor to 10-minute intervals


def resample_magnetic_contact(
    df,
    column_name
):
    """
    Resample one Magnetic Contact time series to
    10-minute intervals and propagate the latest known state.
    """

    result = (
        df
        .set_index("ts")
        .resample("10min")
        .last()
        .ffill()
        .reset_index()
    )

    return result

In [26]:
mc_1_10min = resample_magnetic_contact(
    mc_1,
    "mc_1"
)

mc_2_10min = resample_magnetic_contact(
    mc_2,
    "mc_2"
)

mc_3_10min = resample_magnetic_contact(
    mc_3,
    "mc_3"
)

In [27]:
# 3.3 Integrate the three Magnetic Contact sensors


magnetic_contacts = (
    mc_1_10min
    .copy()
)


for df in [
    mc_2_10min,
    mc_3_10min
]:

    magnetic_contacts = (
        magnetic_contacts
        .merge(
            df,
            on="ts",
            how="outer"
        )
    )


magnetic_contacts = (
    magnetic_contacts
    .sort_values("ts")
    .reset_index(drop=True)
)


sensor_columns = [
    "mc_1",
    "mc_2",
    "mc_3"
]


# Propagate the latest known state after integration.

magnetic_contacts[
    sensor_columns
] = (
    magnetic_contacts[
        sensor_columns
    ]
    .ffill()
)

In [28]:
# 3.4 Restrict to common sensor availability


first_valid_times = {
    column:
        magnetic_contacts.loc[
            magnetic_contacts[
                column
            ].notna(),
            "ts"
        ].min()

    for column in sensor_columns
}


print(
    "First available measurement:"
)

print(
    first_valid_times
)


# Use the latest first-observation timestamp as the
# beginning of the common observation period.

common_start = (
    max(
        first_valid_times.values()
    )
)


magnetic_contacts = (
    magnetic_contacts[
        magnetic_contacts[
            "ts"
        ] >= common_start
    ]
    .copy()
    .reset_index(drop=True)
)

First available measurement:
{'mc_1': Timestamp('2025-08-31 23:20:00'), 'mc_2': Timestamp('2025-08-31 23:30:00'), 'mc_3': Timestamp('2025-08-31 23:40:00')}


In [29]:
# 3.5 Create room-level window variables


magnetic_contacts[
    "open_windows_all"
] = (
    magnetic_contacts[
        sensor_columns
    ]
    .sum(axis=1)
    .astype(int)
)


# This variable is retained for compatibility with the
# subsequent Room 3.9 analysis.

magnetic_contacts[
    "open_windows_dynamic"
] = (
    magnetic_contacts[
        "open_windows_all"
    ]
)

In [30]:
# 3.6 Final validation and export


print(
    "Final shape:",
    magnetic_contacts.shape
)

print("\nTime range:")
print(
    magnetic_contacts["ts"].min(),
    "→",
    magnetic_contacts["ts"].max()
)

print("\nMissing values:")
print(
    magnetic_contacts.isna().sum()
)

print("\nDuplicate timestamps:")
print(
    magnetic_contacts[
        "ts"
    ]
    .duplicated()
    .sum()
)

print("\nUnique values per sensor:")

for column in sensor_columns:

    print(
        column,
        sorted(
            magnetic_contacts[
                column
            ].unique()
        )
    )


print(
    "\nDistribution of open_windows_all:"
)

print(
    magnetic_contacts[
        "open_windows_all"
    ]
    .value_counts()
    .sort_index()
)


display(
    magnetic_contacts.head()
)


output_file = (
    "magnetic_contacts_3_9_clean.csv"
)

magnetic_contacts.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)


print(
    f"\nSaved: {output_file}"
)

Final shape: (44319, 6)

Time range:
2025-08-31 23:40:00 → 2026-07-05 18:00:00

Missing values:
ts                      0
mc_1                    0
mc_2                    0
mc_3                    0
open_windows_all        0
open_windows_dynamic    0
dtype: int64

Duplicate timestamps:
0

Unique values per sensor:
mc_1 [np.float64(0.0), np.float64(1.0)]
mc_2 [np.float64(0.0), np.float64(1.0)]
mc_3 [np.float64(0.0), np.float64(1.0)]

Distribution of open_windows_all:
open_windows_all
0    31060
1     5992
2     4019
3     3248
Name: count, dtype: int64


,ts,mc_1,mc_2,mc_3,open_windows_all,open_windows_dynamic
0,2025-08-31 23:40:00,0.0,0.0,0.0,0,0
1,2025-08-31 23:50:00,0.0,0.0,0.0,0,0
2,2025-09-01 00:00:00,0.0,0.0,0.0,0,0
3,2025-09-01 00:10:00,0.0,0.0,0.0,0,0
4,2025-09-01 00:20:00,0.0,0.0,0.0,0,0



Saved: magnetic_contacts_3_9_clean.csv


## 4. Integration of Room 3.9 Sensor Data

The cleaned datasets from the three sensor systems are combined to construct the final dataset for Room 3.9.

The integration combines:

- Indoor Air Quality measurements,
- People Counter measurements,
- Magnetic Contact measurements.

The IAQ and People Counter datasets are first aligned using their common timestamps. Magnetic Contact information is subsequently added using the corresponding 10-minute timestamps.

Observations without an available window-state measurement are removed, and selected discrete variables are converted to integer format.

The resulting dataset constitutes the final integrated dataset used for the analysis of Room 3.9 and is exported as:

`room_3_9_final.csv`

In [31]:
# SENSOR DATA INTEGRATION — ROOM 3.9


# Load the cleaned datasets produced in the previous sections.

iaq = pd.read_csv(
    "/content/iaq_3_9_clean.csv",
    parse_dates=["Timestamp"]
)

pc = pd.read_csv(
    "/content/people_counter_3_9_clean.csv",
    parse_dates=["ts"]
)

mc = pd.read_csv(
    "/content/magnetic_contacts_3_9_clean.csv",
    parse_dates=["ts"]
)


print("IAQ:", iaq.shape)
print("People Counter:", pc.shape)
print("Magnetic Contacts:", mc.shape)

IAQ: (38556, 7)
People Counter: (44051, 5)
Magnetic Contacts: (44319, 6)


In [32]:
# 4.1 Standardize timestamp representation


# The IAQ dataset uses "Timestamp", while the remaining
# datasets use "ts". Rename it to obtain a common key.

iaq = iaq.rename(
    columns={
        "Timestamp": "ts"
    }
)


# Inspect the temporal coverage of the three datasets.

datasets = {
    "IAQ": iaq,
    "People Counter": pc,
    "Magnetic Contacts": mc
}


for name, df in datasets.items():

    print(
        f"{name}: "
        f"{df['ts'].min()} → "
        f"{df['ts'].max()}"
    )

IAQ: 2025-08-31 21:00:00 → 2026-05-28 16:50:00
People Counter: 2025-08-31 21:00:00 → 2026-07-05 20:50:00
Magnetic Contacts: 2025-08-31 23:40:00 → 2026-07-05 18:00:00


### Temporal Integration

The IAQ and People Counter datasets are combined using an inner join, retaining only timestamps available in both datasets. Window-state information is then added from the Magnetic Contact dataset.

Only the total number of open windows (`open_windows_all`) is retained from the Magnetic Contact data, as this variable is used in the subsequent analysis.

In [33]:
# 4.2 Integrate IAQ and People Counter data


room_3_9 = iaq.merge(
    pc,
    on="ts",
    how="inner"
)


print(
    "Shape after IAQ + People Counter integration:",
    room_3_9.shape
)

display(
    room_3_9.head()
)

Shape after IAQ + People Counter integration: (38323, 11)


,ts,co2,humidity,light_level,pir,temperature,co2_level,line_1_period_in,line_1_period_out,line_1_total_in,line_1_total_out
0,2025-08-31 21:00:00,419.0,41.5,0.0,0.0,29.9,Low,0,0,0,0
1,2025-08-31 21:10:00,419.0,41.5,0.0,0.0,29.9,Low,0,0,0,0
2,2025-08-31 21:20:00,420.0,41.5,0.0,0.0,29.9,Low,0,0,0,0
3,2025-08-31 21:30:00,420.0,41.5,0.0,0.0,29.9,Low,0,0,0,0
4,2025-08-31 21:40:00,418.0,41.5,0.0,0.0,29.9,Low,0,0,0,0


In [34]:
# 4.3 Add Magnetic Contact information


room_3_9 = room_3_9.merge(
    mc[
        [
            "ts",
            "open_windows_all"
        ]
    ],
    on="ts",
    how="left"
)


print(
    "Shape after adding Magnetic Contacts:",
    room_3_9.shape
)

Shape after adding Magnetic Contacts: (38323, 12)


In [35]:
# 4.4 Retain observations with available window-state data

room_3_9_final = (
    room_3_9
    .dropna(
        subset=[
            "open_windows_all"
        ]
    )
    .copy()
    .reset_index(drop=True)
)


print(
    "Shape after restricting to available "
    "Magnetic Contact observations:",
    room_3_9_final.shape
)

Shape after restricting to available Magnetic Contact observations: (38307, 12)


In [36]:
# 4.5 Convert discrete variables to integer representation


integer_columns = [
    "pir",
    "line_1_period_in",
    "line_1_period_out",
    "line_1_total_in",
    "line_1_total_out",
    "open_windows_all"
]


# Ensure PIR remains a binary variable.

room_3_9_final["pir"] = (
    room_3_9_final["pir"]
    .round()
    .clip(0, 1)
)


# Convert discrete sensor variables to integer format.

room_3_9_final[
    integer_columns
] = (
    room_3_9_final[
        integer_columns
    ]
    .round()
    .astype(int)
)

### Final Dataset Validation

Before export, the integrated dataset is checked for missing values, duplicate timestamps and valid PIR states. Its final temporal coverage and descriptive statistics are also inspected to verify consistency with the datasets used in the analysis.

In [37]:
# 4.6 Final validation


print(
    "Final Room 3.9 dataset shape:",
    room_3_9_final.shape
)


print("\nTime range:")
print(
    room_3_9_final["ts"].min(),
    "→",
    room_3_9_final["ts"].max()
)


print("\nMissing values:")
print(
    room_3_9_final.isna().sum()
)


print("\nDuplicate timestamps:")
print(
    room_3_9_final[
        "ts"
    ]
    .duplicated()
    .sum()
)


print("\nUnique PIR values:")
print(
    sorted(
        room_3_9_final[
            "pir"
        ].unique()
    )
)


display(
    room_3_9_final.head()
)


display(
    room_3_9_final.describe(
        include="all"
    )
)

Final Room 3.9 dataset shape: (38307, 12)

Time range:
2025-08-31 23:40:00 → 2026-05-28 16:50:00

Missing values:
ts                   0
co2                  0
humidity             0
light_level          0
pir                  0
temperature          0
co2_level            0
line_1_period_in     0
line_1_period_out    0
line_1_total_in      0
line_1_total_out     0
open_windows_all     0
dtype: int64

Duplicate timestamps:
0

Unique PIR values:
[np.int64(0), np.int64(1)]


,ts,co2,humidity,light_level,pir,temperature,co2_level,line_1_period_in,line_1_period_out,line_1_total_in,line_1_total_out,open_windows_all
0,2025-08-31 23:40:00,416.0,41.0,0.0,0,29.9,Low,0,0,0,0,0
1,2025-08-31 23:50:00,416.0,41.0,0.0,0,29.9,Low,0,0,0,0,0
2,2025-09-01 00:00:00,417.0,41.0,0.0,0,29.8,Low,0,0,0,0,0
3,2025-09-01 00:10:00,416.0,41.0,0.0,0,29.9,Low,0,0,0,0,0
4,2025-09-01 00:20:00,416.0,41.0,0.0,0,29.8,Low,0,0,0,0,0


,ts,co2,humidity,light_level,pir,temperature,co2_level,line_1_period_in,line_1_period_out,line_1_total_in,line_1_total_out,open_windows_all
count,38307,38307.000000,38307.000000,38307.000000,38307.000000,38307.000000,38307,38307.000000,38307.000000,38307.000000,38307.000000,38307.000000
unique,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,Low,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,29363,NaN,NaN,NaN,NaN,NaN
mean,2026-01-13 10:38:21.793405952,490.900910,45.534317,0.416504,0.043047,22.296026,NaN,0.161981,0.158953,14.638082,13.686872,0.528781
min,2025-08-31 23:40:00,378.000000,28.500000,0.000000,0.000000,18.100000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2025-11-07 04:05:00,419.000000,40.000000,0.000000,0.000000,20.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2026-01-13 11:30:00,443.000000,45.500000,0.000000,0.000000,21.400000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2026-03-21 13:55:00,495.000000,50.500000,1.000000,0.000000,24.000000,NaN,0.000000,0.000000,12.000000,9.000000,1.000000
max,2026-05-28 16:50:00,3101.000000,66.000000,3.000000,1.000000,30.400000,NaN,32.000000,40.000000,200.000000,191.000000,3.000000


In [38]:
# 4.7 Export final Room 3.9 dataset


output_file = "room_3_9_final.csv"


room_3_9_final.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)


print(
    f"Saved: {output_file}"
)

print(
    "Final shape:",
    room_3_9_final.shape
)

Saved: room_3_9_final.csv
Final shape: (38307, 12)
